In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

import config
from etl.data_loader import DataLoader

loader = DataLoader()

print("DATA_ROOT:", config.PathConfig.DATA_ROOT)
print("PROCESSED:", config.PathConfig.PROCESSED)
print("RAW:", config.PathConfig.RAW)

In [ ]:
processed_files = sorted(Path(config.PathConfig.PROCESSED).glob("*.parquet"))

pd.DataFrame({
    "file": [p.name for p in processed_files],
    "path": [str(p) for p in processed_files],
    "size_mb": [round(p.stat().st_size / 1024 / 1024, 2) for p in processed_files],
})

In [ ]:
symbol = "BTC/USDT"
timeframe = "4h"

df = loader.get_crypto_kline_data(
    symbol=symbol,
    timeframe=timeframe,
)

df.head()

In [ ]:
def describe_df(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [df[c].notna().sum() for c in df.columns],
        "missing": [df[c].isna().sum() for c in df.columns],
        "missing_pct": [df[c].isna().mean() for c in df.columns],
        "sample_value": [df[c].dropna().iloc[0] if df[c].dropna().shape[0] else None for c in df.columns],
    })

describe_df(df)

In [ ]:
pd.DataFrame({
    "symbol": [symbol],
    "timeframe": [timeframe],
    "start": [df.index.min()],
    "end": [df.index.max()],
    "rows": [len(df)],
    "columns": [list(df.columns)],
})

In [ ]:
matrix = loader.get_crypto_matrix(
    symbols=["BTC/USDT", "ETH/USDT", "SOL/USDT", "BNB/USDT"],
    timeframe="4h",
    columns=["close", "volume", "net_taker_vol"],
)

close = matrix["volume"]
close.tail()

In [ ]:
#资金费率表格查询
fund_rate = loader.get_funding_rate_data()
raw_rate = loader.get_raw_funding_rate_data("BTC/USDT")
display(raw_rate)

In [ ]:
from etl.feature_builder_H import build_crypto_features, FeatureBuilderConfig
from etl.feature_registry import get_feature_definitions


cfg = FeatureBuilderConfig(
    decision_timeframe="4h",
    include_funding=True,
    include_oi=True,
    include_cvd_proxy=True,
    include_sentiment=True,
    include_onchain=True,
)

# features = build_crypto_features(cfg=cfg, save=True)
# display(features)
defs = get_feature_definitions()
display(defs)

In [ ]:
# On-chain / DeFi data preview
def describe_onchain_df(df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [df[c].notna().sum() for c in df.columns],
        "missing": [df[c].isna().sum() for c in df.columns],
        "missing_pct": [round(df[c].isna().mean() * 100, 2) for c in df.columns],
        "sample_value": [df[c].dropna().iloc[0] if df[c].notna().any() else None for c in df.columns],
    })

onchain_tables_processed = loader.list_onchain_tables(layer="processed")
onchain_tables_raw = loader.list_onchain_tables(layer="raw")
onchain_tables_factors = loader.list_onchain_tables(layer="factors")

# display(onchain_tables_processed)
# display(onchain_tables_raw)
# display(onchain_tables_factors)

onchain_daily = loader.get_onchain_data(table="onchain_daily", layer="processed")
onchain_features = loader.get_onchain_data(table="onchain_features", layer="factors")
display(onchain_daily.tail())
display(onchain_features.tail() if onchain_features is not None else None)

if onchain_daily is not None and not onchain_daily.empty:
    display(describe_onchain_df(onchain_daily.reset_index()))
    display(loader.get_latest_onchain_row(table="onchain_daily", layer="processed"))

if onchain_features is not None and not onchain_features.empty:
    display(describe_onchain_df(onchain_features.reset_index()))
    display(loader.get_latest_onchain_row(table="onchain_features", layer="factors"))


In [ ]:
from etl.model_feature_loader import load_crypto_feature_table, split_model_inputs

df = load_crypto_feature_table()
X, meta, feature_cols = split_model_inputs(df, feature_set="market_plus_onchain_v1")
display(feature_cols)